In [1]:
!git clone https://github.com/Break-Through-Tech/Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation.git

Cloning into 'Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 114 (delta 58), reused 9 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 22.74 MiB | 13.46 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [2]:
%cd Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation

/content/Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation


In [3]:
!ls -lh data

total 21M
-rw-r--r-- 1 root root  542 Oct  6 19:08 column_manifest.csv
-rw-r--r-- 1 root root 8.3K Oct  6 19:08 DATA_DICTIONARY.md
-rw-r--r-- 1 root root 2.3K Oct  6 19:08 extract_summary.txt
-rw-r--r-- 1 root root  21M Oct  6 19:08 kidney_waitlist_analytic.csv.gz
-rw-r--r-- 1 root root 2.5K Oct  6 19:08 README.md


In [4]:
!cat data/README.md

# data/ — DUA-governed extracts (LOCAL ONLY)

These CSVs are **row-level** OPTN registry extracts governed by a Data Use
Agreement. Do **not** commit, upload, or share them outside the team.

| folder | cohort | rows | notes |
|---|---|---|---|
| `extract_2015_kidney_only/` | kidney-alone, listed 2015+ (`WL_ORG==KI`) | 494,862 | **primary** — used by analysis + app |
| `extract_2015_ki_kp/` | kidney + kidney-pancreas, 2015+ | 511,390 | includes KP combos |
| `extract_2010_ki_kp/` | kidney + kidney-pancreas, 2010+ | 701,496 | widest window |

Each folder has: `kidney_waitlist_analytic.csv`, `extract_summary.txt`,
`column_manifest.csv`.

Rebuild any of these from the raw STAR file with `extraction/build_analytic_extract.py`.

## Obtaining the raw STAR file

The extracts are built from the OPTN **STAR** (Standard Transplant Analysis and
Research) files — specifically `KIDPAN_DATA.DAT` plus its dictionary
`KIDPAN_DATA.htm`. These are **not** in this repo and are **not** a direct
download.


In [5]:
!cat data/extract_summary.txt

OPTN KIDPAN_DATA -- Analytic Extract Summary
Source file:        KIDPAN_DATA.DAT
Source size:        1.40 GB
Rows read:          1,303,788
Rows kept:          494,862
Columns kept:       40
Listing window:     2015-01-01 onward
Cohort:             kidney-alone waitlist registrations (WL_ORG == KI)
Output:             kidney_waitlist_analytic.csv  (92.9 MB)

Outcome distribution
------------------------------------------------
  transplanted                 234,429  ( 47.4%)
  still_waiting                103,300  ( 20.9%)
  removed_administrative        45,332  (  9.2%)
  transplanted_elsewhere        37,243  (  7.5%)
  removed_too_sick              35,219  (  7.1%)
  died                          34,524  (  7.0%)
  unknown                        4,815  (  1.0%)

Missingness by column (%)
------------------------------------------------
  WLKI                       100.00
  MULTIORG                    97.60
  A2A2B_ELIGIBILITY           92.27
  COMPOSITE_DEATH_DATE        80.53
  PTIME

In [6]:
!cat data/DATA_DICTIONARY.md

# Data Dictionary — OPTN Kidney Waitlist Analytic Extract

Project-specific data dictionary for the analytic extract used in this repo.
Source: OPTN `KIDPAN` STAR file (June 2026 release), **kidney-alone** waitlist
registrations listed **2015-01-01 onward**. This is an *expanded, curated*
subset of the 475-variable STAR file — see the official OPTN dictionary
(`KIDPAN_DATA.htm`) for the full variable list.

- **Cohort:** `WL_ORG == "KI"` and `INIT_DATE ≥ 2015-01-01`
- **Rows:** 494,862 registrations · **Columns:** 34 retained + 5 derived
- **Unit of analysis:** one waitlist *registration* (a patient listed at
  multiple centers appears more than once)

> ⚠️ This file documents the data; it contains **no row-level records**. The
> extracts themselves are DUA-governed and are not distributed. Rebuild them
> with `extraction/build_analytic_extract.py`.

---

## Role legend

| Role | Meaning |
|---|---|
| **ID** | Encrypted linkage key; not modeled |
| **cohort** | Used to define / subset 

In [7]:
import pandas as pd
import numpy as np

file_path = "data/kidney_waitlist_analytic.csv.gz"

df = pd.read_csv(
    file_path,
    compression="gzip",
    low_memory=False
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Rows: 494862
Columns: 38

Column names:
['ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR', 'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'END_CPRA', 'REM_CD', 'DAYSWAIT_CHRON', 'END_STAT', 'INIT_AGE', 'DIALYSIS_DATE', 'END_DATE', 'INIT_DATE', 'ETHCAT', 'PT_CODE', 'DAYSWAIT_ALLOC', 'COMPOSITE_DEATH_DATE', 'REGION', 'WL_ID_CODE', 'PREV_TX', 'TX_DATE', 'DON_TY', 'DIAG_KI', 'MULTIORG', 'ORGAN', 'PSTATUS', 'PTIME', 'TRR_ID_CODE', 'DONOR_ID', 'LISTING_CTR_CODE', 'outcome', 'event_adverse', 'event_transplant', 'censored', 'days_to_event']


In [8]:
manifest = pd.read_csv("data/column_manifest.csv")

print("Manifest rows:", len(manifest))
print(manifest.to_string(index=False))

Manifest rows: 35
              column  present_in_file
             PT_CODE             True
          WL_ID_CODE             True
         TRR_ID_CODE             True
            DONOR_ID             True
              WL_ORG             True
               ORGAN             True
                WLKI             True
           INIT_DATE             True
            END_DATE             True
              REM_CD             True
      DAYSWAIT_CHRON             True
      DAYSWAIT_ALLOC             True
               PTIME             True
             PSTATUS             True
COMPOSITE_DEATH_DATE             True
             TX_DATE             True
              DON_TY             True
            INIT_AGE             True
              GENDER             True
              ETHCAT             True
                 ABO             True
           INIT_CPRA             True
            END_CPRA             True
         ON_DIALYSIS             True
       DIALYSIS_DATE            

In [9]:
print(df.dtypes)

ON_DIALYSIS              object
A2A2B_ELIGIBILITY       float64
GENDER                   object
ABO                      object
BMI_TCR                 float64
FUNC_STAT_TCR           float64
INIT_STAT                 int64
INIT_CPRA               float64
END_CPRA                float64
REM_CD                  float64
DAYSWAIT_CHRON          float64
END_STAT                  int64
INIT_AGE                  int64
DIALYSIS_DATE            object
END_DATE                 object
INIT_DATE                object
ETHCAT                    int64
PT_CODE                   int64
DAYSWAIT_ALLOC          float64
COMPOSITE_DEATH_DATE     object
REGION                    int64
WL_ID_CODE                int64
PREV_TX                  object
TX_DATE                  object
DON_TY                   object
DIAG_KI                 float64
MULTIORG                 object
ORGAN                    object
PSTATUS                 float64
PTIME                   float64
TRR_ID_CODE              object
DONOR_ID

In [10]:
print("\nDataFrame memory usage:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")


DataFrame memory usage:
402.81542015075684 MB


In [11]:
unique_counts = df.nunique(dropna=False).sort_values()

print(unique_counts)

ON_DIALYSIS                  2
GENDER                       2
MULTIORG                     2
event_transplant             2
event_adverse                2
censored                     2
PSTATUS                      3
A2A2B_ELIGIBILITY            3
PREV_TX                      3
ORGAN                        3
DON_TY                       4
END_STAT                     5
INIT_STAT                    5
outcome                      7
ETHCAT                       8
ABO                          8
REGION                      11
REM_CD                      19
FUNC_STAT_TCR               24
DIAG_KI                     70
INIT_AGE                    92
LISTING_CTR_CODE           256
INIT_DATE                 3805
DAYSWAIT_CHRON            3942
days_to_event             3942
PTIME                     3983
COMPOSITE_DEATH_DATE      4102
TX_DATE                   4190
END_DATE                  4196
BMI_TCR                   4765
DAYSWAIT_ALLOC            7380
DIALYSIS_DATE             8954
INIT_CPR

In [12]:
cols_to_check = [
    "ON_DIALYSIS",
    "GENDER",
    "event_adverse",
    "event_transplant",
    "censored",
    "outcome"
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- ON_DIALYSIS ---
ON_DIALYSIS
Y    360617
N    134245
Name: count, dtype: int64

--- GENDER ---
GENDER
M    306408
F    188454
Name: count, dtype: int64

--- event_adverse ---
event_adverse
0    425119
1     69743
Name: count, dtype: int64

--- event_transplant ---
event_transplant
0    260433
1    234429
Name: count, dtype: int64

--- censored ---
censored
0    391562
1    103300
Name: count, dtype: int64

--- outcome ---
outcome
transplanted              234429
still_waiting             103300
removed_administrative     45332
transplanted_elsewhere     37243
removed_too_sick           35219
died                       34524
unknown                     4815
Name: count, dtype: int64


In [13]:
pd.crosstab(
    df["outcome"],
    [
        df["event_adverse"],
        df["event_transplant"],
        df["censored"]
    ]
)

event_adverse               0                      1
event_transplant            0               1      0
censored                    0       1       0      0
outcome                                             
died                        0       0       0  34524
removed_administrative  45332       0       0      0
removed_too_sick            0       0       0  35219
still_waiting               0  103300       0      0
transplanted                0       0  234429      0
transplanted_elsewhere  37243       0       0      0
unknown                  4815       0       0      0

In [14]:
cols_to_check = [
    "ETHCAT",
    "ABO",
    "REGION"
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index())


--- ETHCAT ---
ETHCAT
1      202625
2      144926
4       97884
5       36995
6        3949
7        2522
9        4182
998      1779
Name: count, dtype: int64

--- ABO ---
ABO
A      155594
A1       2398
A1B       226
A2        365
A2B        75
AB      18475
B       74036
O      243693
Name: count, dtype: int64

--- REGION ---
REGION
1     20410
2     59797
3     69493
4     56511
5     81578
6     13816
7     40881
8     26070
9     34543
10    31783
11    59980
Name: count, dtype: int64


In [15]:
numeric_cols = [
    "INIT_AGE",
    "BMI_TCR",
    "INIT_CPRA",
    "DAYSWAIT_CHRON",
    "DAYSWAIT_ALLOC",
    "days_to_event"
]

print(df[numeric_cols].describe().T)

                   count         mean          std   min     25%      50%  \
INIT_AGE        494862.0    51.874533    14.601018  0.00   43.00    54.00   
BMI_TCR         493304.0    64.700077  3141.118209  0.18   24.65    28.53   
INIT_CPRA       352121.0     9.786130    24.799939  0.00    0.00     0.00   
DAYSWAIT_CHRON  494809.0   649.532775   654.238307  0.00  144.00   433.00   
DAYSWAIT_ALLOC  488117.0  1303.998160  1068.064599  0.00  501.00  1083.00   
days_to_event   494803.0   649.541492   654.237636  0.00  144.00   433.00   

                    75%        max  
INIT_AGE          63.00      91.00  
BMI_TCR           32.85  430226.67  
INIT_CPRA          0.00     100.00  
DAYSWAIT_CHRON   965.00    4200.00  
DAYSWAIT_ALLOC  1844.00   15395.00  
days_to_event    965.00    4200.00  


In [16]:
print("BMI < 10:", (df["BMI_TCR"] < 10).sum())
print("BMI > 60:", (df["BMI_TCR"] > 60).sum())
print("BMI > 100:", (df["BMI_TCR"] > 100).sum())
print("BMI > 1000:", (df["BMI_TCR"] > 1000).sum())

print("\nLargest BMI values:")
print(df["BMI_TCR"].nlargest(20).to_string(index=False))

BMI < 10: 240
BMI > 60: 474
BMI > 100: 263
BMI > 1000: 124

Largest BMI values:
430226.67
426875.07
419921.88
412207.69
385397.92
368858.13
360746.58
347047.09
345800.80
335986.16
335986.16
331578.95
331437.83
328034.60
322000.00
316386.91
307719.14
306217.99
298572.53
291921.33


In [17]:
bmi_ranges = pd.cut(
    df["BMI_TCR"],
    bins=[-float("inf"), 10, 15, 18.5, 25, 30, 35, 40, 50, 60, 100, 1000, float("inf")]
)

print(bmi_ranges.value_counts().sort_index())

BMI_TCR
(-inf, 10.0]          240
(10.0, 15.0]         1435
(15.0, 18.5]        11242
(18.5, 25.0]       121008
(25.0, 30.0]       158488
(30.0, 35.0]       125130
(35.0, 40.0]        62758
(40.0, 50.0]        12103
(50.0, 60.0]          426
(60.0, 100.0]         211
(100.0, 1000.0]       139
(1000.0, inf]         124
Name: count, dtype: int64


In [18]:
print("BMI > 100:")
print(df.loc[df["BMI_TCR"] > 100, "INIT_AGE"].describe())

print("\nBMI <= 10:")
print(df.loc[df["BMI_TCR"] <= 10, "INIT_AGE"].describe())

BMI > 100:
count    263.000000
mean      50.703422
std       17.258571
min        1.000000
25%       39.000000
50%       53.000000
75%       64.000000
max       87.000000
Name: INIT_AGE, dtype: float64

BMI <= 10:
count    240.000000
mean      50.600000
std       15.614016
min        1.000000
25%       41.750000
50%       54.000000
75%       62.000000
max       79.000000
Name: INIT_AGE, dtype: float64


In [19]:
age_ranges = pd.cut(
    df["INIT_AGE"],
    bins=[-1, 0, 5, 12, 17, 30, 40, 50, 60, 70, 80, 100]
)

print(age_ranges.value_counts().sort_index())

print("\nSmallest ages:")
print(df["INIT_AGE"].value_counts().sort_index().head(15))

INIT_AGE
(-1, 0]         123
(0, 5]         2620
(5, 12]        3271
(12, 17]       6608
(17, 30]      32067
(30, 40]      60441
(40, 50]      95556
(50, 60]     133779
(60, 70]     129405
(70, 80]      30546
(80, 100]       446
Name: count, dtype: int64

Smallest ages:
INIT_AGE
0     123
1     697
2     718
3     488
4     373
5     344
6     302
7     375
8     424
9     448
10    501
11    562
12    659
13    860
14    947
Name: count, dtype: int64


In [20]:
print("Missing INIT_CPRA:", df["INIT_CPRA"].isna().sum())
print("Missing %:", df["INIT_CPRA"].isna().mean() * 100)

print("\nCPRA summary:")
print(df["INIT_CPRA"].describe())

print("\nMost common CPRA values:")
print(df["INIT_CPRA"].value_counts(dropna=False).head(15))

Missing INIT_CPRA: 142741
Missing %: 28.84460718341679

CPRA summary:
count    352121.000000
mean          9.786130
std          24.799939
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64

Most common CPRA values:
INIT_CPRA
0.00      277468
NaN       142741
0.03         904
16.56        587
0.26         583
100.00       568
99.99        543
56.09        500
2.32         499
50.01        496
0.02         474
36.86        409
99.98        348
29.67        321
21.82        318
Name: count, dtype: int64


In [21]:
comparison = df["DAYSWAIT_CHRON"] == df["days_to_event"]

print("Exactly equal:", comparison.sum())
print("Not equal:", (~comparison).sum())

print("\nRows where both are present but different:")
both_present = df["DAYSWAIT_CHRON"].notna() & df["days_to_event"].notna()

print(
    (df.loc[both_present, "DAYSWAIT_CHRON"] !=
     df.loc[both_present, "days_to_event"]).sum()
)

print("\nDifference summary:")
print(
    (df.loc[both_present, "DAYSWAIT_CHRON"] -
     df.loc[both_present, "days_to_event"]).describe()
)

Exactly equal: 494798
Not equal: 64

Rows where both are present but different:
5

Difference summary:
count    494803.000000
mean         -0.000841
std           0.325585
min        -167.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.000000
dtype: float64


In [22]:
both_present = df["DAYSWAIT_CHRON"].notna() & df["days_to_event"].notna()

diff = (
    df.loc[both_present, "DAYSWAIT_CHRON"]
    - df.loc[both_present, "days_to_event"]
)

print("Non-zero differences:")
print(diff[diff != 0].value_counts().sort_index())

print("\nOutcomes of mismatched rows:")
mismatch = both_present & (df["DAYSWAIT_CHRON"] != df["days_to_event"])
print(df.loc[mismatch, "outcome"].value_counts())

print("\nMissing values:")
print("DAYSWAIT_CHRON missing:", df["DAYSWAIT_CHRON"].isna().sum())
print("days_to_event missing:", df["days_to_event"].isna().sum())

Non-zero differences:
-167.0    1
-143.0    1
-51.0     1
-28.0     1
-27.0     1
Name: count, dtype: int64

Outcomes of mismatched rows:
outcome
transplanted    5
Name: count, dtype: int64

Missing values:
DAYSWAIT_CHRON missing: 53
days_to_event missing: 59


In [23]:
init_date = pd.to_datetime(df["INIT_DATE"])
end_date = pd.to_datetime(df["END_DATE"])

negative_dates = end_date < init_date

print("END_DATE before INIT_DATE:", negative_dates.sum())

print("\nOutcomes:")
print(df.loc[negative_dates, "outcome"].value_counts())

print("\nHow negative are they?")
print((end_date[negative_dates] - init_date[negative_dates]).dt.days.describe())

END_DATE before INIT_DATE: 59

Outcomes:
outcome
died                      58
removed_administrative     1
Name: count, dtype: int64

How negative are they?
count      59.000000
mean     -126.610169
std       643.365157
min     -4810.000000
25%       -17.500000
50%        -6.000000
75%        -2.500000
max        -1.000000
dtype: float64


In [24]:
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": df.isna().mean() * 100
})

missing = missing[missing["Missing_Count"] > 0]
missing = missing.sort_values("Missing_Percent", ascending=False)

print(missing)

                      Missing_Count  Missing_Percent
MULTIORG                     483004        97.603776
A2A2B_ELIGIBILITY            456604        92.268956
COMPOSITE_DEATH_DATE         398512        80.529926
PTIME                        267350        54.025163
DIAG_KI                      265540        53.659404
PSTATUS                      264107        53.369828
TRR_ID_CODE                  264107        53.369828
DONOR_ID                     264107        53.369828
TX_DATE                      264107        53.369828
ORGAN                        264107        53.369828
DON_TY                       264107        53.369828
PREV_TX                      264107        53.369828
INIT_CPRA                    142741        28.844607
END_CPRA                     142448        28.785399
DIALYSIS_DATE                134245        27.127765
REM_CD                       103300        20.874506
DAYSWAIT_ALLOC                 6745         1.363006
FUNC_STAT_TCR                  3975         0.

In [25]:
print("Dialysis date missing by dialysis status:")
print(pd.crosstab(
    df["ON_DIALYSIS"],
    df["DIALYSIS_DATE"].isna(),
    margins=True
))

print("\nTransplant date missing by transplant event:")
print(pd.crosstab(
    df["event_transplant"],
    df["TX_DATE"].isna(),
    margins=True
))

print("\nRemoval code missing by outcome:")
print(pd.crosstab(
    df["outcome"],
    df["REM_CD"].isna()
))

Dialysis date missing by dialysis status:
DIALYSIS_DATE   False    True     All
ON_DIALYSIS                          
N                   0  134245  134245
Y              360617       0  360617
All            360617  134245  494862

Transplant date missing by transplant event:
TX_DATE            False    True     All
event_transplant                        
0                      6  260427  260433
1                 230749    3680  234429
All               230755  264107  494862

Removal code missing by outcome:
REM_CD                   False   True 
outcome                               
died                     34524       0
removed_administrative   45332       0
removed_too_sick         35219       0
still_waiting                0  103300
transplanted            234429       0
transplanted_elsewhere   37243       0
unknown                   4815       0


In [26]:
print("Transplant event = 1 but TX_DATE missing:")
print(
    df.loc[
        (df["event_transplant"] == 1) & (df["TX_DATE"].isna()),
        "outcome"
    ].value_counts()
)

print("\nTransplant event = 0 but TX_DATE exists:")
print(
    df.loc[
        (df["event_transplant"] == 0) & (df["TX_DATE"].notna()),
        "outcome"
    ].value_counts()
)

Transplant event = 1 but TX_DATE missing:
outcome
transplanted    3680
Name: count, dtype: int64

Transplant event = 0 but TX_DATE exists:
outcome
died    6
Name: count, dtype: int64


In [27]:
registrations_per_patient = df["PT_CODE"].value_counts()

print("Patients with 1 registration:", (registrations_per_patient == 1).sum())
print("Patients with 2 registrations:", (registrations_per_patient == 2).sum())
print("Patients with 3+ registrations:", (registrations_per_patient >= 3).sum())
print("Maximum registrations for one patient:", registrations_per_patient.max())

Patients with 1 registration: 349609
Patients with 2 registrations: 52358
Patients with 3+ registrations: 12220
Maximum registrations for one patient: 11


In [28]:
print(registrations_per_patient.value_counts().sort_index())

count
1     349609
2      52358
3       9463
4       2013
5        495
6        168
7         53
8         17
9          6
10         3
11         2
Name: count, dtype: int64


In [29]:
print("Initial Medical Status (INIT_STAT):")
print(df["INIT_STAT"].value_counts(dropna=False).sort_index())

Initial Medical Status (INIT_STAT):
INIT_STAT
4010    348947
4020        19
4050       196
4060        30
4099    145670
Name: count, dtype: int64


In [30]:
print("Functional Status (FUNC_STAT_TCR):")
print(df["FUNC_STAT_TCR"].value_counts(dropna=False).sort_index())

Functional Status (FUNC_STAT_TCR):
FUNC_STAT_TCR
1.0            1
996.0        121
998.0       8700
2010.0       371
2020.0      2936
2030.0      1905
2040.0      8364
2050.0     23923
2060.0     36967
2070.0    111493
2080.0    139478
2090.0    109316
2100.0     35412
4010.0        43
4020.0        23
4030.0        89
4040.0       187
4050.0       157
4060.0       505
4070.0      1029
4080.0      2812
4090.0      2413
4100.0      4642
NaN         3975
Name: count, dtype: int64


In [31]:
print("BMI missing AND Functional Status missing:",
      (df["BMI_TCR"].isna() & df["FUNC_STAT_TCR"].isna()).sum())

print("BMI missing only:",
      (df["BMI_TCR"].isna() & df["FUNC_STAT_TCR"].notna()).sum())

print("Functional Status missing only:",
      (df["BMI_TCR"].notna() & df["FUNC_STAT_TCR"].isna()).sum())

BMI missing AND Functional Status missing: 234
BMI missing only: 1324
Functional Status missing only: 3741


In [32]:
temp_dates = pd.to_datetime(df["INIT_DATE"])

cpra_missing_by_year = (
    df.assign(
        listing_year=temp_dates.dt.year,
        cpra_missing=df["INIT_CPRA"].isna()
    )
    .groupby("listing_year")["cpra_missing"]
    .agg(["count", "sum", "mean"])
)

cpra_missing_by_year["missing_percent"] = (
    cpra_missing_by_year["mean"] * 100
)

print(cpra_missing_by_year[["count", "sum", "missing_percent"]])

              count    sum  missing_percent
listing_year                               
2015          36599      0         0.000000
2016          36956      0         0.000000
2017          37209      0         0.000000
2018          40564      0         0.000000
2019          43223      0         0.000000
2020          37825      0         0.000000
2021          42130      0         0.000000
2022          44442      0         0.000000
2023          47003  37380        79.526839
2024          49635  41129        82.862899
2025          53231  43199        81.153839
2026          26045  21033        80.756383


In [33]:
end_cpra_missing_by_year = (
    df.assign(
        listing_year=temp_dates.dt.year,
        end_cpra_missing=df["END_CPRA"].isna()
    )
    .groupby("listing_year")["end_cpra_missing"]
    .agg(["count", "sum", "mean"])
)

end_cpra_missing_by_year["missing_percent"] = (
    end_cpra_missing_by_year["mean"] * 100
)

print(end_cpra_missing_by_year[["count", "sum", "missing_percent"]])

              count    sum  missing_percent
listing_year                               
2015          36599    718         1.961802
2016          36956   1092         2.954865
2017          37209   1899         5.103604
2018          40564   3313         8.167340
2019          43223   5144        11.901071
2020          37825   6411        16.949108
2021          42130  10557        25.058153
2022          44442  16732        37.649071
2023          47003  25530        54.315682
2024          49635  26491        53.371613
2025          53231  29186        54.828953
2026          26045  15375        59.032444


In [34]:
both_cpra = df["INIT_CPRA"].notna() & df["END_CPRA"].notna()

print("Rows with both CPRA values:", both_cpra.sum())

diff = df.loc[both_cpra, "END_CPRA"] - df.loc[both_cpra, "INIT_CPRA"]

print("\nChange in CPRA:")
print(diff.describe())

Rows with both CPRA values: 303297

Change in CPRA:
count    303297.000000
mean         15.606852
std          30.373371
min        -100.000000
25%           0.000000
50%           0.000000
75%          17.820000
max         100.000000
dtype: float64


In [35]:
print("CPRA increased:", (diff > 0).sum())
print("CPRA stayed same:", (diff == 0).sum())
print("CPRA decreased:", (diff < 0).sum())

CPRA increased: 125365
CPRA stayed same: 162466
CPRA decreased: 15466


In [36]:
listing_year_counts = (
    pd.to_datetime(df["INIT_DATE"])
    .dt.year
    .value_counts()
    .sort_index()
)

print(listing_year_counts)

INIT_DATE
2015    36599
2016    36956
2017    37209
2018    40564
2019    43223
2020    37825
2021    42130
2022    44442
2023    47003
2024    49635
2025    53231
2026    26045
Name: count, dtype: int64


In [37]:
outcome_by_year = pd.crosstab(
    pd.to_datetime(df["INIT_DATE"]).dt.year,
    df["outcome"],
    normalize="index"
) * 100

print(outcome_by_year.round(2))

outcome     died  removed_administrative  removed_too_sick  still_waiting  \
INIT_DATE                                                                   
2015       11.59                   14.78             11.14           0.75   
2016       10.81                   13.58             10.51           1.17   
2017       10.67                   12.67              9.57           1.77   
2018       10.07                   11.58              9.45           2.60   
2019        9.82                   11.02              9.40           4.07   
2020        9.27                   10.46              8.81           6.57   
2021        7.58                   10.17              8.17          10.80   
2022        5.88                    9.38              7.35          16.90   
2023        4.49                    8.11              5.74          25.79   
2024        3.23                    5.64              4.09          38.66   
2025        1.62                    2.90              1.79          59.29   

In [38]:
followup_by_year = (
    df.assign(
        listing_year=pd.to_datetime(df["INIT_DATE"]).dt.year
    )
    .groupby("listing_year")
    .agg(
        registrations=("WL_ID_CODE", "count"),
        median_days_observed=("days_to_event", "median"),
        mean_days_observed=("days_to_event", "mean"),
        censored_percent=("censored", "mean")
    )
)

followup_by_year["censored_percent"] *= 100

print(followup_by_year.round(2))

              registrations  median_days_observed  mean_days_observed  \
listing_year                                                            
2015                  36599                 737.0              963.66   
2016                  36956                 700.0              917.80   
2017                  37209                 654.0              878.21   
2018                  40564                 629.0              848.91   
2019                  43223                 610.0              808.19   
2020                  37825                 563.0              754.95   
2021                  42130                 538.0              704.05   
2022                  44442                 506.0              640.24   
2023                  47003                 470.0              547.58   
2024                  49635                 474.0              434.07   
2025                  53231                 269.0              266.49   
2026                  26045                  74.0  

In [39]:
init_date = pd.to_datetime(df["INIT_DATE"])
end_date = pd.to_datetime(df["END_DATE"])

print("Earliest Listing Date:", init_date.min())
print("Latest Listing Date:", init_date.max())

print("\nEarliest End Date:", end_date.min())
print("Latest End Date:", end_date.max())

Earliest Listing Date: 2015-01-01 00:00:00
Latest Listing Date: 2026-06-30 00:00:00

Earliest End Date: 2002-12-15 00:00:00
Latest End Date: 2026-07-03 00:00:00


In [40]:
adverse_by_year = (
    df.assign(
        listing_year=pd.to_datetime(df["INIT_DATE"]).dt.year
    )
    .groupby("listing_year")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

adverse_by_year["adverse_percent"] = adverse_by_year["mean"] * 100

print(adverse_by_year[["count", "sum", "adverse_percent"]].round(2))

              count   sum  adverse_percent
listing_year                              
2015          36599  8321            22.74
2016          36956  7880            21.32
2017          37209  7529            20.23
2018          40564  7915            19.51
2019          43223  8306            19.22
2020          37825  6841            18.09
2021          42130  6637            15.75
2022          44442  5880            13.23
2023          47003  4809            10.23
2024          49635  3632             7.32
2025          53231  1819             3.42
2026          26045   174             0.67


In [41]:
resolved = df[df["censored"] == 0].copy()

resolved_adverse_by_year = (
    resolved.assign(
        listing_year=pd.to_datetime(resolved["INIT_DATE"]).dt.year
    )
    .groupby("listing_year")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

resolved_adverse_by_year["adverse_percent"] = (
    resolved_adverse_by_year["mean"] * 100
)

print(
    resolved_adverse_by_year[
        ["count", "sum", "adverse_percent"]
    ].round(2)
)

              count   sum  adverse_percent
listing_year                              
2015          36324  8321            22.91
2016          36524  7880            21.57
2017          36552  7529            20.60
2018          39509  7915            20.03
2019          41462  8306            20.03
2020          35340  6841            19.36
2021          37579  6637            17.66
2022          36930  5880            15.92
2023          34883  4809            13.79
2024          30446  3632            11.93
2025          21673  1819             8.39
2026           4340   174             4.01


In [42]:
age_bins = pd.cut(
    df["INIT_AGE"],
    bins=[-1, 17, 29, 39, 49, 59, 69, 79, 100],
    labels=[
        "0-17",
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70-79",
        "80+"
    ]
)

age_adverse = (
    df.assign(age_group=age_bins)
    .groupby("age_group", observed=False)["event_adverse"]
    .agg(["count", "sum", "mean"])
)

age_adverse["adverse_percent"] = age_adverse["mean"] * 100

print(age_adverse[["count", "sum", "adverse_percent"]].round(2))

            count    sum  adverse_percent
age_group                                
0-17        12622    253             2.00
18-29       27847   1184             4.25
30-39       57300   3832             6.69
40-49       91712   9832            10.72
50-59      130674  19748            15.11
60-69      135066  26806            19.85
70-79       38912   7951            20.43
80+           729    137            18.79


In [43]:
gender_adverse = (
    df.groupby("GENDER")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

gender_adverse["adverse_percent"] = gender_adverse["mean"] * 100

print(gender_adverse[["count", "sum", "adverse_percent"]].round(2))

         count    sum  adverse_percent
GENDER                                
F       188454  24584            13.05
M       306408  45159            14.74


In [44]:
ethnicity_labels = {
    1: "White",
    2: "Black",
    4: "Hispanic",
    5: "Asian",
    6: "American Indian",
    7: "Pacific Islander",
    9: "Multiracial",
    998: "Unknown"
}

eth_adverse = (
    df.assign(
        ethnicity=df["ETHCAT"].map(ethnicity_labels)
    )
    .groupby("ethnicity")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

eth_adverse["adverse_percent"] = eth_adverse["mean"] * 100

print(
    eth_adverse[
        ["count", "sum", "adverse_percent"]
    ].sort_values("adverse_percent", ascending=False).round(2)
)

                   count    sum  adverse_percent
ethnicity                                       
American Indian     3949    672            17.02
Pacific Islander    2522    367            14.55
White             202625  29406            14.51
Black             144926  20604            14.22
Hispanic           97884  13208            13.49
Asian              36995   4890            13.22
Multiracial         4182    525            12.55
Unknown             1779     71             3.99


In [45]:
abo_adverse = (
    df.groupby("ABO")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

abo_adverse["adverse_percent"] = abo_adverse["mean"] * 100

print(
    abo_adverse[
        ["count", "sum", "adverse_percent"]
    ].sort_values("adverse_percent", ascending=False).round(2)
)

      count    sum  adverse_percent
ABO                                
A2      365     57            15.62
O    243693  36572            15.01
B     74036  10997            14.85
A2B      75     10            13.33
A    155594  19940            12.82
A1     2398    287            11.97
AB    18475   1858            10.06
A1B     226     22             9.73


In [46]:
dialysis_adverse = (
    df.groupby("ON_DIALYSIS")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

dialysis_adverse["adverse_percent"] = dialysis_adverse["mean"] * 100

print(
    dialysis_adverse[
        ["count", "sum", "adverse_percent"]
    ].round(2)
)

              count    sum  adverse_percent
ON_DIALYSIS                                
N            134245  13798            10.28
Y            360617  55945            15.51


In [47]:
cpra_group = pd.cut(
    df["INIT_CPRA"],
    bins=[-0.01, 0, 20, 50, 80, 98, 100],
    labels=[
        "0",
        "0.01-20",
        "20.01-50",
        "50.01-80",
        "80.01-98",
        "98.01-100"
    ]
)

cpra_adverse = (
    df.assign(cpra_group=cpra_group)
    .groupby("cpra_group", observed=False)["event_adverse"]
    .agg(["count", "sum", "mean"])
)

cpra_adverse["adverse_percent"] = cpra_adverse["mean"] * 100

print(cpra_adverse[["count", "sum", "adverse_percent"]].round(2))

             count    sum  adverse_percent
cpra_group                                
0           277468  51537            18.57
0.01-20      23711   3072            12.96
20.01-50     17575   2244            12.77
50.01-80     14600   1953            13.38
80.01-98     11112   1406            12.65
98.01-100     7655   1241            16.21


In [48]:
older = df[
    pd.to_datetime(df["INIT_DATE"]).dt.year <= 2022
].copy()

cpra_group_old = pd.cut(
    older["INIT_CPRA"],
    bins=[-0.01, 0, 20, 50, 80, 98, 100],
    labels=[
        "0",
        "0.01-20",
        "20.01-50",
        "50.01-80",
        "80.01-98",
        "98.01-100"
    ]
)

cpra_adverse_old = (
    older.assign(cpra_group=cpra_group_old)
    .groupby("cpra_group", observed=False)["event_adverse"]
    .agg(["count", "sum", "mean"])
)

cpra_adverse_old["adverse_percent"] = cpra_adverse_old["mean"] * 100

print(
    cpra_adverse_old[
        ["count", "sum", "adverse_percent"]
    ].round(2)
)

             count    sum  adverse_percent
cpra_group                                
0           275137  51282            18.64
0.01-20      13670   2480            18.14
20.01-50      9717   1769            18.21
50.01-80      8585   1602            18.66
80.01-98      7002   1155            16.50
98.01-100     4837   1021            21.11


In [49]:
region_adverse = (
    df.groupby("REGION")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

region_adverse["adverse_percent"] = region_adverse["mean"] * 100

print(
    region_adverse[
        ["count", "sum", "adverse_percent"]
    ].sort_values("adverse_percent", ascending=False).round(2)
)

        count    sum  adverse_percent
REGION                               
1       20410   3460            16.95
4       56511   8791            15.56
5       81578  12231            14.99
2       59797   8916            14.91
3       69493   9721            13.99
10      31783   4342            13.66
9       34543   4680            13.55
11      59980   8057            13.43
6       13816   1722            12.46
8       26070   3091            11.86
7       40881   4732            11.58


In [50]:
func_adverse = (
    df.groupby("FUNC_STAT_TCR")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

func_adverse["adverse_percent"] = func_adverse["mean"] * 100

print(
    func_adverse[
        ["count", "sum", "adverse_percent"]
    ].sort_index().round(2)
)

                count    sum  adverse_percent
FUNC_STAT_TCR                                
1.0                 1      0             0.00
996.0             121      7             5.79
998.0            8700   1438            16.53
2010.0            371    103            27.76
2020.0           2936    709            24.15
2030.0           1905    443            23.25
2040.0           8364   1534            18.34
2050.0          23923   4061            16.98
2060.0          36967   6654            18.00
2070.0         111493  18267            16.38
2080.0         139478  18937            13.58
2090.0         109316  13468            12.32
2100.0          35412   3875            10.94
4010.0             43     11            25.58
4020.0             23      5            21.74
4030.0             89     15            16.85
4040.0            187     13             6.95
4050.0            157     15             9.55
4060.0            505     23             4.55
4070.0           1029     24      

In [51]:
status_adverse = (
    df.groupby("INIT_STAT")["event_adverse"]
    .agg(["count", "sum", "mean"])
)

status_adverse["adverse_percent"] = status_adverse["mean"] * 100

print(
    status_adverse[
        ["count", "sum", "adverse_percent"]
    ].sort_index().round(2)
)

            count    sum  adverse_percent
INIT_STAT                                
4010       348947  49944            14.31
4020           19      0             0.00
4050          196     35            17.86
4060           30      9            30.00
4099       145670  19755            13.56


In [52]:
bmi_valid = df[df["BMI_TCR"].between(10, 60)].copy()

bmi_group = pd.cut(
    bmi_valid["BMI_TCR"],
    bins=[10, 18.5, 25, 30, 35, 40, 50, 60],
    labels=[
        "10-18.5",
        "18.5-25",
        "25-30",
        "30-35",
        "35-40",
        "40-50",
        "50-60"
    ]
)

bmi_adverse = (
    bmi_valid.assign(bmi_group=bmi_group)
    .groupby("bmi_group", observed=False)["event_adverse"]
    .agg(["count", "sum", "mean"])
)

bmi_adverse["adverse_percent"] = bmi_adverse["mean"] * 100

print(bmi_adverse[["count", "sum", "adverse_percent"]].round(2))

            count    sum  adverse_percent
bmi_group                                
10-18.5     12677    999             7.88
18.5-25    121008  15322            12.66
25-30      158488  22859            14.42
30-35      125130  19280            15.41
35-40       62758   9223            14.70
40-50       12103   1595            13.18
50-60         426     67            15.73


In [53]:
wait_group = pd.cut(
    df["days_to_event"],
    bins=[-0.1, 90, 180, 365, 730, 1095, 1825, float("inf")],
    labels=[
        "0-3 months",
        "3-6 months",
        "6-12 months",
        "1-2 years",
        "2-3 years",
        "3-5 years",
        "5+ years"
    ]
)

wait_adverse = (
    df.assign(wait_group=wait_group)
    .groupby("wait_group", observed=False)["event_adverse"]
    .agg(["count", "sum", "mean"])
)

wait_adverse["adverse_percent"] = wait_adverse["mean"] * 100

print(
    wait_adverse[
        ["count", "sum", "adverse_percent"]
    ].round(2)
)

              count    sum  adverse_percent
wait_group                                 
0-3 months    90279   4753             5.26
3-6 months    53049   3975             7.49
6-12 months   80670   8859            10.98
1-2 years    101752  16327            16.05
2-3 years     65891  12992            19.72
3-5 years     70054  15379            21.95
5+ years      33108   7400            22.35


In [54]:
print("Total Days on Waitlist (DAYSWAIT_CHRON):")
print(df["DAYSWAIT_CHRON"].describe())

print("\nAllocation Priority Days (DAYSWAIT_ALLOC):")
print(df["DAYSWAIT_ALLOC"].describe())

print("\nAllocation days greater than chronological wait days:")
valid = df["DAYSWAIT_ALLOC"].notna() & df["DAYSWAIT_CHRON"].notna()

print(
    (df.loc[valid, "DAYSWAIT_ALLOC"] >
     df.loc[valid, "DAYSWAIT_CHRON"]).sum()
)

print("\nDifference:")
print(
    (
        df.loc[valid, "DAYSWAIT_ALLOC"] -
        df.loc[valid, "DAYSWAIT_CHRON"]
    ).describe()
)

Total Days on Waitlist (DAYSWAIT_CHRON):
count    494809.000000
mean        649.532775
std         654.238307
min           0.000000
25%         144.000000
50%         433.000000
75%         965.000000
max        4200.000000
Name: DAYSWAIT_CHRON, dtype: float64

Allocation Priority Days (DAYSWAIT_ALLOC):
count    488117.000000
mean       1303.998160
std        1068.064599
min           0.000000
25%         501.000000
50%        1083.000000
75%        1844.000000
max       15395.000000
Name: DAYSWAIT_ALLOC, dtype: float64

Allocation days greater than chronological wait days:
334551

Difference:
count    488077.000000
mean        649.158186
std         966.991109
min       -3720.000000
25%           0.000000
50%         311.000000
75%         893.000000
max       15356.000000
dtype: float64


In [55]:
dialysis_date = pd.to_datetime(df["DIALYSIS_DATE"], errors="coerce")
listing_date = pd.to_datetime(df["INIT_DATE"], errors="coerce")

dialysis_before_listing = (
    dialysis_date.notna() &
    listing_date.notna() &
    (dialysis_date < listing_date)
)

print("Dialysis started before listing:", dialysis_before_listing.sum())

print("\nDays from dialysis start to listing:")
print(
    (listing_date[dialysis_before_listing] -
     dialysis_date[dialysis_before_listing])
    .dt.days
    .describe()
)

Dialysis started before listing: 318009

Days from dialysis start to listing:
count    318009.000000
mean        906.035870
std        1009.114285
min           1.000000
25%         273.000000
50%         561.000000
75%        1167.000000
max       15356.000000
dtype: float64


In [56]:
valid = (
    dialysis_before_listing &
    df["DAYSWAIT_ALLOC"].notna() &
    df["DAYSWAIT_CHRON"].notna()
)

extra_allocation_days = (
    df.loc[valid, "DAYSWAIT_ALLOC"] -
    df.loc[valid, "DAYSWAIT_CHRON"]
)

prelisting_dialysis_days = (
    listing_date[valid] -
    dialysis_date[valid]
).dt.days

difference = extra_allocation_days - prelisting_dialysis_days

print("Rows compared:", valid.sum())

print("\nExtra allocation days:")
print(extra_allocation_days.describe())

print("\nPre-listing dialysis days:")
print(prelisting_dialysis_days.describe())

print("\nDifference between the two:")
print(difference.describe())

Rows compared: 317966

Extra allocation days:
count    317966.000000
mean        959.839291
std        1034.726992
min           1.000000
25%         288.000000
50%         607.000000
75%        1269.000000
max       15356.000000
dtype: float64

Pre-listing dialysis days:
count    317966.000000
mean        906.022905
std        1009.111267
min           1.000000
25%         273.000000
50%         561.000000
75%        1167.000000
max       15356.000000
dtype: float64

Difference between the two:
count    317966.000000
mean         53.816386
std         271.345961
min          -1.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        7672.000000
dtype: float64


In [57]:
print("Exact match:",
      (difference == 0).sum())

print("Within 1 day:",
      (difference.abs() <= 1).sum())

print("Within 30 days:",
      (difference.abs() <= 30).sum())

print("More than 365 days different:",
      (difference.abs() > 365).sum())

Exact match: 295750
Within 1 day: 295767
Within 30 days: 296169
More than 365 days different: 15086


In [58]:
unusual = valid & (difference.abs() > 365)

print("Unusual rows:", unusual.sum())

print("\nOutcomes:")
print(df.loc[unusual, "outcome"].value_counts())

print("\nListing years:")
print(
    pd.to_datetime(df.loc[unusual, "INIT_DATE"])
    .dt.year
    .value_counts()
    .sort_index()
)

Unusual rows: 15086

Outcomes:
outcome
transplanted              7619
still_waiting             3835
transplanted_elsewhere    1202
removed_too_sick           951
removed_administrative     887
died                       543
unknown                     49
Name: count, dtype: int64

Listing years:
INIT_DATE
2015     237
2016     314
2017     333
2018     479
2019     722
2020     760
2021    1164
2022    1754
2023    2513
2024    2655
2025    2814
2026    1341
Name: count, dtype: int64


In [59]:
abo_transplant = (
    df.groupby("ABO")["event_transplant"]
    .agg(["count", "sum", "mean"])
)

abo_transplant["transplant_percent"] = abo_transplant["mean"] * 100

print(
    abo_transplant[
        ["count", "sum", "transplant_percent"]
    ].sort_values("transplant_percent", ascending=False).round(2)
)

      count     sum  transplant_percent
ABO                                    
A1B     226     157               69.47
A2B      75      50               66.67
AB    18475   11568               62.61
A1     2398    1461               60.93
A2      365     203               55.62
A    155594   83072               53.39
B     74036   32752               44.24
O    243693  105166               43.16


In [60]:
transplanted_only = df[df["event_transplant"] == 1].copy()

abo_wait = (
    transplanted_only
    .groupby("ABO")["days_to_event"]
    .agg(["count", "median", "mean"])
)

print(abo_wait.round(2))

      count  median    mean
ABO                        
A     83072   221.0  420.90
A1     1461   207.0  392.80
A1B     157   137.0  226.92
A2      203   284.0  411.18
A2B      50   129.5  242.98
AB    11568   170.0  330.36
B     32752   285.0  527.51
O    105166   276.0  532.05


In [61]:
region_transplant = (
    df.groupby("REGION")["event_transplant"]
    .agg(["count", "sum", "mean"])
)

region_transplant["transplant_percent"] = region_transplant["mean"] * 100

print(
    region_transplant[
        ["count", "sum", "transplant_percent"]
    ].sort_values("transplant_percent", ascending=False).round(2)
)

        count    sum  transplant_percent
REGION                                  
8       26070  14501               55.62
6       13816   7672               55.53
10      31783  17358               54.61
7       40881  21058               51.51
9       34543  17669               51.15
11      59980  28305               47.19
3       69493  32207               46.35
2       59797  27321               45.69
5       81578  35652               43.70
1       20410   8688               42.57
4       56511  23998               42.47


In [62]:
region_wait = (
    transplanted_only
    .groupby("REGION")["days_to_event"]
    .agg(["count", "median", "mean"])
)

print(region_wait.sort_values("median").round(2))

        count  median    mean
REGION                       
4       23998   211.0  434.40
10      17358   216.0  416.09
3       32207   218.0  445.40
9       17669   223.0  473.76
11      28305   231.0  442.62
8       14501   234.0  414.55
7       21058   243.0  456.87
2       27321   276.0  512.67
5       35652   301.0  574.87
6        7672   325.0  518.61
1        8688   400.5  658.07


In [63]:
region_censored = (
    df.groupby("REGION")["censored"]
    .agg(["count", "sum", "mean"])
)

region_censored["still_waiting_percent"] = region_censored["mean"] * 100

print(
    region_censored[
        ["count", "sum", "still_waiting_percent"]
    ].sort_values("still_waiting_percent", ascending=False).round(2)
)

        count    sum  still_waiting_percent
REGION                                     
5       81578  21858                  26.79
1       20410   5451                  26.71
11      59980  12958                  21.60
6       13816   2894                  20.95
9       34543   7081                  20.50
4       56511  11093                  19.63
7       40881   7825                  19.14
3       69493  13129                  18.89
2       59797  11269                  18.85
8       26070   4533                  17.39
10      31783   5209                  16.39


In [64]:
region_outcomes = pd.crosstab(
    df["REGION"],
    df["outcome"],
    normalize="index"
) * 100

print(region_outcomes.round(2))

outcome  died  removed_administrative  removed_too_sick  still_waiting  \
REGION                                                                   
1        9.41                    7.77              7.54          26.71   
2        7.52                   10.16              7.39          18.85   
3        6.61                   10.71              7.38          18.89   
4        6.52                   11.54              9.03          19.63   
5        9.44                    7.91              5.55          26.79   
6        4.30                    7.22              8.16          20.95   
7        5.71                    9.47              5.86          19.14   
8        4.66                    8.80              7.20          17.39   
9        7.75                    9.28              5.80          20.50   
10       5.61                    7.95              8.05          16.39   
11       5.87                    7.26              7.56          21.60   

outcome  transplanted  transplanted_e

In [65]:
# ============================================================
# FINAL REMAINING EDA CHECKS
# Kidney Waitlist Analytic Dataset
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("1. FINAL DATASET STRUCTURE")
print("=" * 80)

print("Rows / waitlist registrations:", len(df))
print("Columns:", df.shape[1])
print("Unique patients (PT_CODE):", df["PT_CODE"].nunique())
print("Unique waitlist registrations (WL_ID_CODE):", df["WL_ID_CODE"].nunique())
print("Duplicate waitlist IDs:", df["WL_ID_CODE"].duplicated().sum())


# ============================================================
# 2. LISTING-TIME MODEL FEATURES: MISSINGNESS
# ============================================================

print("\n" + "=" * 80)
print("2. LISTING-TIME FEATURE COMPLETENESS")
print("=" * 80)

feature_names = {
    "INIT_AGE": "Age at Listing",
    "GENDER": "Sex",
    "ETHCAT": "Race/Ethnicity",
    "ABO": "Blood Type",
    "INIT_CPRA": "Initial CPRA",
    "ON_DIALYSIS": "On Dialysis at Listing",
    "BMI_TCR": "Body Mass Index",
    "FUNC_STAT_TCR": "Functional Status",
    "INIT_STAT": "Initial Medical Status",
    "REGION": "OPTN Region"
}

listing_features = list(feature_names.keys())

feature_missing = pd.DataFrame({
    "Feature": [feature_names[c] for c in listing_features],
    "Column": listing_features,
    "Missing_Count": [df[c].isna().sum() for c in listing_features],
    "Missing_Percent": [df[c].isna().mean() * 100 for c in listing_features]
})

print(feature_missing.round(2).to_string(index=False))

complete_cases = df[listing_features].notna().all(axis=1).sum()

print("\nRegistrations with all 10 listing-time features present:",
      complete_cases)

print("Complete-case percentage:",
      round(complete_cases / len(df) * 100, 2), "%")


# ============================================================
# 3. FINAL DATA-QUALITY FLAGS
# ============================================================

print("\n" + "=" * 80)
print("3. DATA-QUALITY FLAGS")
print("=" * 80)

init_date = pd.to_datetime(df["INIT_DATE"], errors="coerce")
end_date = pd.to_datetime(df["END_DATE"], errors="coerce")

print("Age at Listing (INIT_AGE) < 0:",
      (df["INIT_AGE"] < 0).sum())

print("Age at Listing (INIT_AGE) > 100:",
      (df["INIT_AGE"] > 100).sum())

print("\nInitial CPRA (INIT_CPRA) outside 0-100:",
      ((df["INIT_CPRA"] < 0) | (df["INIT_CPRA"] > 100)).sum())

print("End CPRA (END_CPRA) outside 0-100:",
      ((df["END_CPRA"] < 0) | (df["END_CPRA"] > 100)).sum())

print("\nBody Mass Index (BMI_TCR) <= 10:",
      (df["BMI_TCR"] <= 10).sum())

print("Body Mass Index (BMI_TCR) > 60:",
      (df["BMI_TCR"] > 60).sum())

print("Body Mass Index (BMI_TCR) > 100:",
      (df["BMI_TCR"] > 100).sum())

print("Body Mass Index (BMI_TCR) > 1000:",
      (df["BMI_TCR"] > 1000).sum())

print("\nEnd Date before Listing Date:",
      (end_date < init_date).sum())

print("Suspicious Functional Status (FUNC_STAT_TCR = 1):",
      (df["FUNC_STAT_TCR"] == 1).sum())


# ============================================================
# 4. REMAINING CATEGORICAL / DESCRIPTIVE VARIABLES
# ============================================================

print("\n" + "=" * 80)
print("4. REMAINING CATEGORICAL VARIABLES")
print("=" * 80)

remaining_categorical = {
    "MULTIORG": "Multi-organ Listing Flag",
    "A2A2B_ELIGIBILITY": "A2/A2B-to-B Eligibility",
    "PREV_TX": "Previous Transplant Indicator",
    "DON_TY": "Donor Type",
    "ORGAN": "Actually Transplanted Organ",
    "PSTATUS": "Patient Status After Transplant"
}

for col, name in remaining_categorical.items():
    print(f"\n--- {name} ({col}) ---")
    print(df[col].value_counts(dropna=False))


# ============================================================
# 5. A2/A2B ELIGIBILITY BY BLOOD TYPE
# ============================================================

print("\n" + "=" * 80)
print("5. A2/A2B ELIGIBILITY BY BLOOD TYPE")
print("=" * 80)

a2_present = df["A2A2B_ELIGIBILITY"].notna()

print(
    pd.crosstab(
        df["ABO"],
        a2_present,
        margins=True
    ).rename(
        columns={
            False: "Eligibility Missing",
            True: "Eligibility Recorded"
        }
    )
)


# ============================================================
# 6. OUTCOME TIMING
# ============================================================

print("\n" + "=" * 80)
print("6. OBSERVED TIME BY OUTCOME")
print("=" * 80)

outcome_time = (
    df.groupby("outcome")["days_to_event"]
    .agg(["count", "median", "mean", "min", "max"])
    .sort_values("median")
)

print(outcome_time.round(2))


# ============================================================
# 7. TRANSPLANT-TIME FIELD CONSISTENCY
# ============================================================

print("\n" + "=" * 80)
print("7. TRANSPLANT-TIME FIELD CONSISTENCY")
print("=" * 80)

print("\nTransplant Date (TX_DATE) availability by outcome:")

print(
    pd.crosstab(
        df["outcome"],
        df["TX_DATE"].notna()
    ).rename(
        columns={
            False: "TX_DATE Missing",
            True: "TX_DATE Present"
        }
    )
)

print("\nDonor Type (DON_TY) availability by outcome:")

print(
    pd.crosstab(
        df["outcome"],
        df["DON_TY"].notna()
    ).rename(
        columns={
            False: "DON_TY Missing",
            True: "DON_TY Present"
        }
    )
)

print("\nDeath Date (COMPOSITE_DEATH_DATE) availability by outcome:")

print(
    pd.crosstab(
        df["outcome"],
        df["COMPOSITE_DEATH_DATE"].notna()
    ).rename(
        columns={
            False: "Death Date Missing",
            True: "Death Date Present"
        }
    )
)


# ============================================================
# 8. KIDNEY DIAGNOSIS CODES
# ============================================================

print("\n" + "=" * 80)
print("8. MOST COMMON KIDNEY DIAGNOSIS CODES")
print("=" * 80)

print(df["DIAG_KI"].value_counts(dropna=False).head(15))


# ============================================================
# 9. TRANSPLANT DONOR TYPE
# ============================================================

print("\n" + "=" * 80)
print("9. DONOR TYPE AMONG TRANSPLANTED REGISTRATIONS")
print("=" * 80)

print(
    df.loc[
        df["event_transplant"] == 1,
        "DON_TY"
    ].value_counts(dropna=False)
)


# ============================================================
# 10. OUTCOME SUMMARY BY LISTING CENTER
# ============================================================

print("\n" + "=" * 80)
print("10. LISTING-CENTER VARIATION")
print("=" * 80)

center_summary = (
    df.groupby("LISTING_CTR_CODE")
    .agg(
        registrations=("WL_ID_CODE", "count"),
        adverse_events=("event_adverse", "sum"),
        transplants=("event_transplant", "sum"),
        still_waiting=("censored", "sum")
    )
)

center_summary["adverse_percent"] = (
    center_summary["adverse_events"] /
    center_summary["registrations"] * 100
)

center_summary["transplant_percent"] = (
    center_summary["transplants"] /
    center_summary["registrations"] * 100
)

center_summary["still_waiting_percent"] = (
    center_summary["still_waiting"] /
    center_summary["registrations"] * 100
)

print("Total listing centers:",
      len(center_summary))

print("\nCenter registration-count summary:")
print(center_summary["registrations"].describe().round(2))

# Only compare centers with at least 500 registrations
large_centers = center_summary[
    center_summary["registrations"] >= 500
].copy()

print("\nCenters with at least 500 registrations:",
      len(large_centers))

print("\nHighest adverse-event rates among centers with >=500 registrations:")
print(
    large_centers[
        ["registrations", "adverse_percent"]
    ]
    .sort_values("adverse_percent", ascending=False)
    .head(10)
    .round(2)
)

print("\nLowest adverse-event rates among centers with >=500 registrations:")
print(
    large_centers[
        ["registrations", "adverse_percent"]
    ]
    .sort_values("adverse_percent")
    .head(10)
    .round(2)
)

print("\nHighest transplant rates among centers with >=500 registrations:")
print(
    large_centers[
        ["registrations", "transplant_percent"]
    ]
    .sort_values("transplant_percent", ascending=False)
    .head(10)
    .round(2)
)

print("\nLowest transplant rates among centers with >=500 registrations:")
print(
    large_centers[
        ["registrations", "transplant_percent"]
    ]
    .sort_values("transplant_percent")
    .head(10)
    .round(2)
)


# ============================================================
# 11. CENTER-LEVEL RATE DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("11. CENTER-LEVEL RATE DISTRIBUTION")
print("=" * 80)

print(
    large_centers[
        [
            "adverse_percent",
            "transplant_percent",
            "still_waiting_percent"
        ]
    ].describe().round(2)
)


# ============================================================
# 12. FINAL TARGET BALANCE
# ============================================================

print("\n" + "=" * 80)
print("12. FINAL TARGET BALANCE")
print("=" * 80)

target_summary = pd.DataFrame({
    "Count": df["event_adverse"].value_counts().sort_index(),
    "Percent": (
        df["event_adverse"].value_counts(normalize=True)
        .sort_index() * 100
    )
})

target_summary.index = [
    "No recorded adverse event",
    "Died / Removed too sick"
]

print(target_summary.round(2))


# ============================================================
# 13. FINAL OUTCOME DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("13. FINAL OUTCOME DISTRIBUTION")
print("=" * 80)

final_outcomes = pd.DataFrame({
    "Count": df["outcome"].value_counts(),
    "Percent": df["outcome"].value_counts(normalize=True) * 100
})

print(final_outcomes.round(2))


print("\n" + "=" * 80)
print("FINAL EDA BATCH COMPLETE")
print("=" * 80)

1. FINAL DATASET STRUCTURE
Rows / waitlist registrations: 494862
Columns: 38
Unique patients (PT_CODE): 414187
Unique waitlist registrations (WL_ID_CODE): 494862
Duplicate waitlist IDs: 0

2. LISTING-TIME FEATURE COMPLETENESS
               Feature        Column  Missing_Count  Missing_Percent
        Age at Listing      INIT_AGE              0             0.00
                   Sex        GENDER              0             0.00
        Race/Ethnicity        ETHCAT              0             0.00
            Blood Type           ABO              0             0.00
          Initial CPRA     INIT_CPRA         142741            28.84
On Dialysis at Listing   ON_DIALYSIS              0             0.00
       Body Mass Index       BMI_TCR           1558             0.31
     Functional Status FUNC_STAT_TCR           3975             0.80
Initial Medical Status     INIT_STAT              0             0.00
           OPTN Region        REGION              0             0.00

Registrations 